# 5. Action Classification — Object Detection

Trains a YOLO11m model to classify volleyball actions (spike, block, dig, set, etc.)
Detections from this model are linked to player tracks in the backend.

| | |
|---|---|
| **Dataset** | [volleyball-actions v3](https://universe.roboflow.com/actions-players/volleyball-actions/dataset/3) |
| **Model** | `yolo11m.pt` |
| **Task** | Object detection (multi-class: volleyball actions) |
| **Output** | `runs/detect/action-classification-v3/weights/best.pt` |

> **Why `yolo11m` (medium)?** Action classification involves more classes with visually
> similar poses. The medium model has significantly more capacity than nano/small
> to learn fine-grained distinctions between actions.
>
> **Note:** The Roboflow dataset uses `yolov8` export format, which is fully compatible
> with YOLO11 — the annotation format is identical.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────

# Model — medium gives more capacity for multi-class action distinctions
MODEL = "yolo11m.pt"       # yolo11n/s/m/l/x

# Training
EPOCHS      = 150
IMGSZ       = 640
BATCH       = 16
LR0         = 0.01
LRF         = 0.005
COS_LR      = True
MIXUP       = 0.1           # augmentation: blends two images, helps with action confusion
FREEZE      = 0             # freeze first N backbone layers (0 = off; try 10 for small datasets)
PATIENCE    = 20
SAVE_PERIOD = 10
WORKERS     = 4

# Data
DATASET_DIR  = "datasets/action_classification"
RF_WORKSPACE = "actions-players"
RF_PROJECT   = "volleyball-actions"
RF_VERSION   = 3
RF_FORMAT    = "yolov8"     # yolov8 format is fully compatible with yolo11 training

# Run identity
RUN_PROJECT   = "runs/detect"
RUN_NAME      = "action-classification-v3"
WANDB_PROJECT = "tobio"

In [ ]:
import os
import wandb
from dotenv import load_dotenv
from roboflow import Roboflow
from ultralytics import YOLO

from utils import setup_device, print_metrics

load_dotenv()
DEVICE = setup_device()

if WANDB_PROJECT:
    wandb.init(
        project=WANDB_PROJECT,
        name=RUN_NAME,
        config={
            "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ,
            "batch": BATCH, "lr0": LR0, "lrf": LRF, "cos_lr": COS_LR,
            "mixup": MIXUP, "freeze": FREEZE, "dataset": f"{RF_PROJECT} v{RF_VERSION}",
        },
    )

In [ ]:
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
dataset = (
    rf.workspace(RF_WORKSPACE)
    .project(RF_PROJECT)
    .version(RF_VERSION)
    .download(RF_FORMAT, location=DATASET_DIR)
)
print(f"Dataset location: {dataset.location}")

In [ ]:
model = YOLO(MODEL)

train_results = model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    optimizer="Adam",
    lr0=LR0,
    lrf=LRF,
    cos_lr=COS_LR,
    mixup=MIXUP,
    freeze=FREEZE,
    patience=PATIENCE,
    project=RUN_PROJECT,
    name=RUN_NAME,
    save=True,
    save_period=SAVE_PERIOD,
    exist_ok=True,
    verbose=True,
    plots=True,
    val=True,
)

In [ ]:
# augment=True enables Test-Time Augmentation (TTA): multi-scale + flip ensemble
metrics = model.val(augment=True)
print_metrics(metrics)

# tensorboard --logdir runs/detect

In [ ]:
onnx_path = model.export(format="onnx")
print(f"ONNX model saved to: {onnx_path}")

if WANDB_PROJECT:
    wandb.finish()